### Deep learning model training.


In [3]:
import time
import torch
import random
import numpy as np
import pandas as pd
import torch.nn as nn
from glob import glob
from notebooks import config
from torchinfo import summary
import matplotlib.pyplot as plt
import torch.nn.functional as F
from utils.imgShow import imsShow
from utils.utils import read_scenes
from torchvision.transforms import v2
from utils.data_aug import GaussianNoise
from torchmetrics.classification import MulticlassJaccardIndex, BinaryAccuracy
from utils.dataloader import SceneArraySet, PatchSet
from model.seg import unet, deeplabv3plus, deeplabv3plus_mobilev2, watnet



### dataset loading

In [4]:
patch_size = 512        ## patch size setting
learning_rate = 1e-4     
batch_size = 8   
device = torch.device('cuda:0')  
model_name = 'unet'


In [5]:
### secene and truth pairwise data
## traset
paths_scene = sorted(glob(config.dir_scene + '/*_nor.tif'))
paths_truth = [config.dir_truth + 
               '/' + 
               path.split('/')[-1].replace('_6Bands', '').replace('_nor', '_Truth') \
               for path in paths_scene]
paths_scene_tra = paths_scene[20:]
paths_truth_tra = paths_truth[20:]
## valset
paths_patch_valset = sorted(glob(config.dir_valset + '/*'))
print(f'train scenes: {len(paths_scene_tra)}, vali patch: {len(paths_patch_valset)}')


train scenes: 75, vali patch: 275


In [ ]:
scenes_arr, truths_arr = read_scenes(paths_scene_tra, paths_truth_tra) 
scene_truth_ls =  list(zip(scenes_arr, truths_arr))
scene_truth_ls = [np.concatenate([scene, truth[:, :, np.newaxis]], axis=-1) 
                                for scene, truth in scene_truth_ls] 


In [ ]:
transforms_tra = v2.Compose([
      v2.ToImage(),
      v2.RandomCrop(size=(patch_size, patch_size), padding = 50, fill=0),
      v2.RandomApply([v2.RandomRotation(degrees=15)], p=0.3),
      v2.RandomHorizontalFlip(p=0.3),
      v2.RandomVerticalFlip(p=0.3),
      GaussianNoise(mean=0, sigma_max=0.1, p=0.3)    
       ])
transforms_val = v2.Compose([
      v2.ToDtype(torch.float32),
       ])


In [ ]:
tra_dset = SceneArraySet(scene_truth_list = scene_truth_ls, transforms = transforms_tra)
val_dset = PatchSet(paths_valset=paths_patch_valset)


In [ ]:
tra_loader = torch.utils.data.DataLoader(tra_dset, 
                                         batch_size=4, 
                                         shuffle=True, 
                                         num_workers=10)
val_loader = torch.utils.data.DataLoader(val_dset, 
                                         batch_size=4)


#### Model training

In [ ]:
### check model
model = unet(num_bands=6, num_classes=2)
# model = watnet(num_bands=6)
# model = deeplabv3plus(num_bands=6)
# model = deeplabv3plus_mobilev2(num_bands=6)
summary(model, input_size=(1,6,256,256), device='cpu') 


Layer (type:depth-idx)                   Output Shape              Param #
unet                                     [1, 1, 256, 256]          --
├─ModuleList: 1-1                        --                        --
│    └─dsample: 2-1                      [1, 16, 128, 128]         384
│    │    └─AvgPool2d: 3-1               [1, 6, 128, 128]          --
│    │    └─Sequential: 3-2              [1, 32, 128, 128]         1,824
│    │    └─Sequential: 3-3              [1, 32, 128, 128]         384
│    │    └─Sequential: 3-4              [1, 16, 128, 128]         560
│    └─dsample: 2-2                      [1, 16, 64, 64]           768
│    │    └─AvgPool2d: 3-5               [1, 16, 64, 64]           --
│    │    └─Sequential: 3-6              [1, 64, 64, 64]           9,408
│    │    └─Sequential: 3-7              [1, 64, 64, 64]           768
│    │    └─Sequential: 3-8              [1, 16, 64, 64]           1,072
│    └─dsample: 2-3                      [1, 32, 32, 32]           1,53

In [ ]:
### create loss and optimizer
loss_bce = nn.BCELoss()     
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)   


In [ ]:

def show_result(model, patch, truth):
    model.eval()
    device = next(model.parameters()).device
    patch, truth = patch.unsqueeze(0).to(device), truth.to(device)
    pred = model(patch) 
    ## convert to numpy and plot
    patch = patch[0].to('cpu').detach().numpy().transpose(1,2,0)            
    pred = pred[0].to('cpu').detach().numpy()
    truth = truth.to('cpu').detach().numpy()
    imsShow([patch, pred, truth],   
            clip_list = (2,0,0),
            img_name_list=['patch', 'pred', 'truth'], 
            color_bands_list=([3,2,2], 0, 0),                    
            figsize=(9, 2.5))
    plt.tight_layout() 
    plt.show()   



In [ ]:
'''------train loops------'''
def train_loops(model, loss_fn, 
                    optimizer, 
                    tra_loader, 
                    val_loader,  
                    epoches, 
                    device, 
                    lr_scheduler=None):
    loss_tra_loops, miou_tra_loops, oa_tra_loops = [], [], []
    loss_val_loops, miou_val_loops, oa_val_loops = [], [], []
    model = model.to(device)
    size_tra_loader = len(tra_loader)
    size_val_loader = len(val_loader)
    best_miou = 0.93
    epoches_i = []
    for epoch in range(epoches):
        start = time.time()
        loss_tra, loss_val = 0, 0
        '''-----train the model-----'''
        oa_tra = BinaryAccuracy().to(device)
        miou_tra = MulticlassJaccardIndex(num_classes=2, average="macro").to(device)
        model.train()   # training mode for dropout and batchnorm
        for x_batch, y_batch in tra_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            pred = model(x_batch)
            loss = loss_fn(pred, y_batch)
            loss.backward()
            optimizer.step()
            pred = (pred > 0.5).float()
            miou_tra.update(pred, y_batch.long())
            oa_tra.update(pred, y_batch.long())
            loss_tra += loss.item()
        miou_tra_global = miou_tra.compute()
        oa_tra_global = oa_tra.compute()
        loss_tra_global = loss_tra/size_tra_loader
        miou_tra.reset(); oa_tra.reset()

        '''----- validation the model: time consuming -----'''
        oa_val = BinaryAccuracy().to(device)
        miou_val = MulticlassJaccardIndex(num_classes=2, average="macro").to(device)
        model.eval()
        if (epoch+1) % 2 == 0: 
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                with torch.no_grad():
                    pred = model(x_batch.float())
                    loss = loss_fn(pred, y_batch.float())
                pred = (pred > 0.5).float()
                miou_val.update(pred, y_batch.long())
                oa_val.update(pred, y_batch.long())
                loss_val += loss.item()
            miou_val_global = miou_val.compute()
            oa_val_global = oa_val.compute()
            loss_val_global = loss_val/size_val_loader
            miou_val.reset(); oa_val.reset()
            loss_tra_loops.append(loss_tra_global); miou_tra_loops.append(miou_tra_global.item()); oa_tra_loops.append(oa_tra_global.item())
            loss_val_loops.append(loss_val_global); miou_val_loops.append(miou_val_global.item()); oa_val_loops.append(oa_val_global.item())
            epoches_i.append(epoch)
            print(f'Ep{epoch}: tra-> Loss:{loss_tra_global:.3f},Oa:{oa_tra_global:.3f},Miou:{miou_tra_global:.3f}, '
                    f'val-> Loss:{loss_val_global:.3f},Oa:{oa_val_global:.3f}, Miou:{miou_val_global:.3f},time:{time.time()-start:.1f}s')
            ## save the best model
            if miou_val_global.item() > best_miou:
                best_miou = miou_val_global.item()       ## update best miou
                torch.save(model.state_dict(), f'model/trained/{model_name}_0{str(round(best_miou*10000))}.pth')
        else: 
            print(f'Ep{epoch}: tra-> Loss:{loss_tra_global:.3f},Oa:{oa_tra_global:.3f},Miou:{miou_tra_global:.3f}, \
                                time:{time.time()-start:.1f}s')
        if lr_scheduler: lr_scheduler.step(miou_tra_global)    ## if use lr_scheduler like ReduceLROnPlateau
        if (epoch+1)%20 == 0:
            sam_index = random.randrange(len(val_dset))
            patch_val, truth_val = val_dset[sam_index]
            show_result(model=model, patch=patch_val.float(), truth=truth_val.float())
    metrics = {'epoch':epoches_i, 'tra_loss':loss_tra_loops, 'tra_oa': oa_tra_loops, 'tra_miou': miou_tra_loops,
                'val_loss': loss_val_loops, 'val_oa': oa_val_loops, 'val_miou': miou_val_loops}
    return metrics  


In [ ]:
device = torch.device('cuda') 
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
metrics = train_loops(model=model, 
              epoches=200,  
              loss_fn=loss_bce, 
              optimizer=optimizer, 
              tra_loader=tra_loader,   
              val_loader=val_loader,  
              device=device)     


NameError: name 'torch' is not defined

In [ ]:
# ## model saving
# path_save = f'model/trained/{model_name}_weights.pth'
# torch.save(model.state_dict(), path_save)   # save weights of the trained model 
# # model.load_state_dict(torch.load(path_save, weights_only=True))  # load the weights of the trained model
# ## metrics saving
# path_metrics = f'model/trained/{model_name}_metrics.csv'
# metrics_df = pd.DataFrame(metrics)
# metrics_df.to_csv(path_metrics, index=False, sep=',')
